## Использование Gamac на реальных данных
Описание реальных задач приведено в [документации](https://github.com/ITMO-CODE-AI/GaMAC/blob/develop/docs/CASE_RU.md)

### Импортируем нужные библиотеки

In [1]:
import os
import sys
sys.path.append('../../')

import numpy as np
import pandas as pd
from PIL import Image

from gamac.autoclustering import Gamac
from gamac.estimation.internal import Internal

### 1. Данные промышленных логов (семпл)
Кластеризация логов направлена на автоматическое выделение групп записей, связанных с общими источниками событий, такими как прикладные процессы, системные события или среды исполнения. Основная цель — обнаружить скрытые паттерны в данных, которые позволяют косвенно определить природу возникновения логов, даже если их формат не содержит явных указаний на источник.

Для получения данных нужно выгрузить их из Minio, данные лежат в: datasets/data/synthetic_logs_1kk.csv

И переложить из корня в папку data/

Примечание: выполнение по этому датасету проходит очень долгое время в связи с его объемом

In [2]:
!pip install minio

In [3]:
# загрзука данных из MinIO
import os
from minio import Minio
os.environ["SSL_CERT_FILE"] = 'all-data/public.crt'

In [4]:
def download_data_without_creds(data):
    client = Minio(
    '94.124.179.195:9000',
    secure=True
    )
    data = client.fget_object(
        'datasets',
        f'data/{data}',
        f'all-data/{data}'
    )

In [5]:
download_data_without_creds('synthetic_logs_1kk.csv')

In [6]:
# Импортируем датафрейм
data = pd.read_csv('all-data/synthetic_logs_1kk.csv', na_values=['missing'])
data['timestamp'] = pd.to_datetime(data['timestamp'], errors='coerce')

In [7]:
data.head(5)

,Unnamed: 0,object,agent,subject,environment,source,result,result code,tag,log level,label,traceback,protocol,action,property,timestamp
0,0,NaN,NaN,thread,NaN,24.149.203.18,[<:subject:> <:source:>] File does not exist:<...,NaN,NaN,error,NaN,event interval expired,NaN,NaN,lifetime 05:00,2025-04-18 10:08:58
1,1,dfs.FSNamesystem,ocsp.digicert.com:80,out block blk 8855628190418004805,NaN,transfer block blk -1132082380757283113 to 10....,<:action:>* NameSystem.addStoredBlock: blockMa...,NaN,27052.0,WARN,NaN,Redundant addStoredBlock request received for ...,NaN,Access denied,power/control problem,2025-03-28 10:08:58
2,2,spoolsv.exe *64,news.17173.com:80,thread,NaN,transfer block blk -1009207079038502874 to 10....,link errors remain current,NaN,6708.0,-1,state_change.unavailable,Redundant addStoredBlock request received for ...,NaN,NaN,power/control problem,2025-04-19 10:08:58
3,3,chrome.exe,pubads.g.doubleclick.net:443,node-139,NaN,84.92.64.33,"<:action:>, <:NUM:> bytes <:*:> <:*:> sent, <:...",NaN,NaN,NaN,NaN,timed,NaN,close,power/control problem,2025-04-14 10:08:58
4,4,partition,mini.cpc.sogou.com:80,thread,NaN,85.64.10.104,critical,NaN,385417.0,-1,status,must start with /,NaN,NaN,power/control problem,2025-04-16 10:08:58


In [8]:
gamac = Gamac(iter_limit=100, target_measures={Internal.BR})

result = gamac.run(table=data)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 1.1026804447174072s, {'BR': -16.44806543893915} ===
=== MEASURES 0.01060795783996582s, {'BR': -12.456415287817997} ===
=== ALGO 3.031867265701294s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 7, 'max_iter': 20, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 1.9073486328125e-05s, FailedRun, {'name': 'MeanShift', 'bandwidth': 0.1416808612149756, 'max_iter': 5, 'tol': 0.0001} ===
=== ALGO 41.14652729034424s, FailedRun, {'name': 'DBSCAN', 'eps': 1.4608874205047306, 'eps_sq': 2.1341920553889655, 'min_samples': 12} ===
=== ALGO 22.90702772140503s, FailedRun, {'name': 'HDBSCAN', 'min_cluster_size': 9, 'min_samples': 7} ===
=== MEASURES 0.008957147598266602s, {'BR': -11.895881968258244} ===
=== ALGO 224.63210582733154s, SuccessRun, {'name': 'Birch', 'threshold': 0.7799884730999638, 'branching_factor': 45, 'n_clusters': 9} ===
=== MEASURES 0.008044719696044922s, {'BR': -11.790844441260644} ===
=== ALGO 0.07422828674316406s, SuccessRun, {'name': 'KMeans', 'n_clusters': 8, '

In [9]:
# Получение топ-50 меток по лучшему классификатору
print(result.labels[:50])

[ 0  8 14 12 14  8  3  6 12  5 13 14  7  1  5  5 14  5  5 12  6  1 10  5
 11  8  3 12  0  5  6  1 10 11  5  1 14  8  5 14  5  2  5 11 14  1  5 14
  3  5]


### 2. Данные по анализу цветов RGB-изображений
Кластеризация цветов RGB-изображений направлена на автоматическое группирование пикселей со схожими цветовыми значениями в отдельные кластеры, где каждый кластер представляет собой усреднённый цвет или доминирующий оттенок из соответствующей группы. Основная цель — упростить представление изображения за счёт выделения ключевых цветовых паттернов, что может быть полезно для задач сжатия данных, сегментации объектов, снижения шума или анализа цветовой палитры.

In [10]:
# Импортируем картинки
images = []

# Получаем список файлов в директории
for filename in os.listdir('../../data/images/'):
    if 'png' not in filename:
        continue
    file_path = os.path.join('../../data/images/', filename)

    # Пытаемся открыть файл как изображение
    with Image.open(file_path) as img:
        images.append(img.copy())

images[:5]

[<PIL.Image.Image image mode=RGBA size=600x600>,
 <PIL.Image.Image image mode=RGBA size=651x888>,
 <PIL.Image.Image image mode=RGBA size=3433x4606>,
 <PIL.Image.Image image mode=RGBA size=114x102>,
 <PIL.Image.Image image mode=RGBA size=744x1052>]

### Запустим подбор

In [11]:
# Для кластеризации передадим константный пустой текст
gamac = Gamac(iter_limit=30)
empty_texts = ['' for i in range(len(images))]

result = gamac.run(image=images, text=empty_texts)

/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== Started CVI prediction ===
=== CVI prediction iteration 1/2 ====
=== CVI prediction iteration 2/2 ====
=== Picked SYM in 12.066900968551636s ===
=== MEASURES 0.3256947994232178s, {'SYM': 0.0004269955482819506} ===
=== MEASURES 0.050510406494140625s, {'SYM': 0.007340950293377901} ===
=== ALGO 1.6903407573699951s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 4, 'max_iter': 20, 'init': 'k-means++', 'tol': 0.0001} ===
=== ALGO 1.9311904907226562e-05s, FailedRun, {'name': 'MeanShift', 'bandwidth': 0.15436615777741444, 'max_iter': 5, 'tol': 0.0001} ===
=== MEASURES 0.12204813957214355s, {'SYM': 0.0} ===
=== ALGO 0.794041633605957s, SuccessRun, {'name': 'DBSCAN', 'eps': 1.2320443188029702, 'eps_sq': 1.5179332034946749, 'min_samples': 5} ===
=== ALGO 0.7455964088439941s, FailedRun, {'name': 'HDBSCAN', 'min_cluster_size': 5, 'min_samples': 7} ===
=== ALGO 3.2698981761932373s, FailedRun, {'name': 'Birch', 'threshold': 0.27648330189043835, 'branching_factor': 70, 'n_clusters': 14} ==

In [12]:
# Метки по датасету
print(result.labels)

[ 8 10  8  3  6  1  5  4  4  8  4  4 10 10  5  3  4  9  1  3  8  8  3  1
  8  4  4  8  8  5  8  8  7 10  6  3  5  3  9  6  4  1  2  7 10  9  7  1
  1  7  3  9  9 10  4  8  8  6  5  2 10  1  2  3  2  7  4  6  4  1 10  2
  9  5  2  9  6  8  4  5  1  9  4  4  3 10  5  5  4  2  3  3  2  4  4 10
  5  5  9  8  4  7  1  8  8  1  5  8  4 10  4  8  5  3  4  3  1  4  5  4
  2  2  9  3  8  1  4  2  5  9  8  8  6  3 10  3 10  7 10  8  1  1  2 10
  8  5 10  6  4  5  1  3  9  7  6  7  5  5  4  3  4  4  3  4  9  9  3  4
  8  8  8  2  5 10  9 10  1  2  2  6  8  7  3  9  2  5  1  4  5  7 10  6
  4  6  3  2  5  1  6  4  6  7  3  1  1  9 10  8  4  2  4  6  9  4  4  7
  8  4  1  4  6  9  6  5  4  6  3  4  7 10  8  9  3  2  9 10 10  5  6  4
  4  4  5 10  4  4  9  8  3  7  5  5 10  3  2 10  6  0  2  5  5  2  4  5
  9  6  4  4 10 10  3  3  4 10  1  9 10  5  8  9  2 10 10  9  2  8  8 10
  5  8  6  2  9  2  5  1  5  5  4  9  4  2  7  4  8  6 10 10  1  9 10  6
  8  0  5  1  6  4  6  8  2  5  2 10  8  3  3 10  2

### Аналогичные действия с GaMAC, установленным через pip

In [1]:
!pip install -U --extra-index-url https://test.pypi.org/simple/ Gamac --extra-index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://pypi.org/simple, https://test.pypi.org/simple/, https://download.pytorch.org/whl/cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 6.4 MB/s eta 0:00:0000:0100:01


In [6]:
import os
import sys

import numpy as np
import pandas as pd
from PIL import Image

from gamac import autoclustering
from gamac import estimation

In [7]:
import os
from minio import Minio
os.environ["SSL_CERT_FILE"] = 'all-data/public.crt'
def download_data_without_creds(data):
    client = Minio(
    '94.124.179.195:9000',
    secure=True
    )
    data = client.fget_object(
        'datasets',
        f'data/{data}',
        f'all-data/{data}'
    )
download_data_without_creds('synthetic_logs_1kk.csv')
# Импортируем датафрейм
data = pd.read_csv('all-data/synthetic_logs_1kk.csv', na_values=['missing'])
data['timestamp'] = pd.to_datetime(data['timestamp'], errors='coerce')
data.head(5)

,Unnamed: 0,object,agent,subject,environment,source,result,result code,tag,log level,label,traceback,protocol,action,property,timestamp
0,0,NaN,NaN,thread,NaN,24.149.203.18,[<:subject:> <:source:>] File does not exist:<...,NaN,NaN,error,NaN,event interval expired,NaN,NaN,lifetime 05:00,2025-04-18 10:08:58
1,1,dfs.FSNamesystem,ocsp.digicert.com:80,out block blk 8855628190418004805,NaN,transfer block blk -1132082380757283113 to 10....,<:action:>* NameSystem.addStoredBlock: blockMa...,NaN,27052.0,WARN,NaN,Redundant addStoredBlock request received for ...,NaN,Access denied,power/control problem,2025-03-28 10:08:58
2,2,spoolsv.exe *64,news.17173.com:80,thread,NaN,transfer block blk -1009207079038502874 to 10....,link errors remain current,NaN,6708.0,-1,state_change.unavailable,Redundant addStoredBlock request received for ...,NaN,NaN,power/control problem,2025-04-19 10:08:58
3,3,chrome.exe,pubads.g.doubleclick.net:443,node-139,NaN,84.92.64.33,"<:action:>, <:NUM:> bytes <:*:> <:*:> sent, <:...",NaN,NaN,NaN,NaN,timed,NaN,close,power/control problem,2025-04-14 10:08:58
4,4,partition,mini.cpc.sogou.com:80,thread,NaN,85.64.10.104,critical,NaN,385417.0,-1,status,must start with /,NaN,NaN,power/control problem,2025-04-16 10:08:58


In [8]:
gamac = autoclustering.Gamac(iter_limit=100, target_measures={estimation.internal.Internal.BR})

result = gamac.run(table=data)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 1.1157617568969727s, {'BR': -16.44806280726097} ===
=== MEASURES 0.009341955184936523s, {'BR': -14.626955715340738} ===
=== ALGO 2.3537065982818604s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 2, 'max_iter': 20, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 2.0265579223632812e-05s, FailedRun, {'name': 'MeanShift', 'bandwidth': 0.33740411433523265, 'max_iter': 5, 'tol': 0.0001} ===
=== ALGO 34.414361000061035s, FailedRun, {'name': 'DBSCAN', 'eps': 1.0504753027357927, 'eps_sq': 1.1034983616578553, 'min_samples': 12} ===
=== ALGO 63.111104011535645s, FailedRun, {'name': 'HDBSCAN', 'min_cluster_size': 5, 'min_samples': 5} ===
=== MEASURES 0.01004934310913086s, {'BR': -11.692344446095483} ===
=== ALGO 135.4413707256317s, SuccessRun, {'name': 'Birch', 'threshold': 0.749319266971823, 'branching_factor': 22, 'n_clusters': 14} ===
=== MEASURES 0.00537872314453125s, {'BR': -13.687040946218175} ===
=== ALGO 0.05107903480529785s, SuccessRun, {'name': 'KMeans', 'n_clusters':

In [9]:
# Получение топ-50 меток по лучшему классификатору
print(result.labels[:50])

[ 2 14  9  6  9 14 14 13  0  4  5  9 12  1  3  4  9  3  3  6 13  1  9  4
  7 14 14  0  2  3 13  1  6  7  3  1  9 14  4  9  3  7  3  7  8  1  3  9
 14  4]


In [10]:
# Импортируем картинки
images = []

# Получаем список файлов в директории
for filename in os.listdir('../../data/images/'):
    if 'png' not in filename:
        continue
    file_path = os.path.join('../../data/images/', filename)

    # Пытаемся открыть файл как изображение
    with Image.open(file_path) as img:
        images.append(img.copy())

images[:5]

[<PIL.Image.Image image mode=RGBA size=600x600>,
 <PIL.Image.Image image mode=RGBA size=651x888>,
 <PIL.Image.Image image mode=RGBA size=3433x4606>,
 <PIL.Image.Image image mode=RGBA size=114x102>,
 <PIL.Image.Image image mode=RGBA size=744x1052>]

In [11]:
gamac = Gamac(iter_limit=30)
empty_texts = ['' for i in range(len(images))]

result = gamac.run(image=images, text=empty_texts)

/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== Started CVI prediction ===
=== CVI prediction iteration 1/2 ====
=== CVI prediction iteration 2/2 ====
=== Picked SYM in 13.202826023101807s ===
=== MEASURES 0.3189883232116699s, {'SYM': 0.000511937877054524} ===
=== ALGO 2.4685208797454834s, FailedRun, {'name': 'BisectingKMeans', 'n_clusters': 12, 'max_iter': 20, 'init': 'k-means++', 'tol': 0.0001} ===
=== ALGO 2.1696090698242188e-05s, FailedRun, {'name': 'MeanShift', 'bandwidth': 0.7076722791579713, 'max_iter': 5, 'tol': 0.0001} ===
=== MEASURES 0.3281221389770508s, {'SYM': 0.0} ===
=== ALGO 2.0018069744110107s, SuccessRun, {'name': 'DBSCAN', 'eps': 1.309090159370299, 'eps_sq': 1.713717045360155, 'min_samples': 14} ===
=== ALGO 0.5287821292877197s, FailedRun, {'name': 'HDBSCAN', 'min_cluster_size': 9, 'min_samples': 8} ===
=== ALGO 4.424066066741943s, FailedRun, {'name': 'Birch', 'threshold': 0.4485855606022975, 'branching_factor': 66, 'n_clusters': 13} ===
=== MEASURES 0.1943802833557129s, {'SYM': 0.013563322590189343} ===
=== A

In [12]:
# Метки по датасету
print(result.labels)

[4 3 5 3 0 5 3 3 3 4 3 3 6 2 3 3 3 2 5 3 4 4 3 5 4 3 3 4 4 4 4 4 4 6 0 3 3
 3 2 0 3 5 6 4 2 2 5 5 5 4 3 2 2 2 3 6 4 0 3 4 2 3 6 3 6 4 3 0 3 5 6 6 2 3
 4 2 0 4 3 3 5 6 3 3 3 6 3 3 3 6 3 3 5 3 3 2 4 3 2 4 3 4 5 4 6 5 3 4 5 6 5
 4 3 3 3 3 5 3 3 3 6 6 2 3 4 5 3 6 3 2 4 4 0 3 6 3 6 4 6 4 5 5 6 6 4 3 6 0
 3 3 5 3 2 4 0 1 3 3 3 3 3 3 3 3 2 2 3 3 4 4 4 6 3 6 2 2 5 6 2 0 4 4 3 2 6
 3 5 3 3 4 2 0 3 0 3 6 3 5 0 3 0 4 3 5 3 2 6 4 3 6 3 0 2 3 3 4 6 3 5 5 5 2
 0 3 3 0 3 3 4 6 4 2 3 6 2 6 6 3 0 3 3 3 3 6 3 5 2 4 3 4 3 3 6 3 4 6 0 3 2
 3 3 2 3 3 2 0 3 3 2 6 3 3 3 2 5 2 6 3 4 2 6 6 6 2 6 2 4 2 3 4 0 6 2 6 3 5
 0 3 3 2 3 6 4 3 2 0 6 6 5 2 6 0 4 3 3 5 0 3 0 4 6 3 6 6 6 3 3 6 6 0 4 4 6
 0 3 2 4 6 5 3 6 2 5 6 2 6 4 2 3 3 0 6 3 2 4 2 2 3 3 6 6 3 2 3 3 2 2 4 5 5
 4 0 2 5 3 2 2 3 2 6 3 0 3 6 3 0 3 0 0 6 4 6 3 3 5 4 3 3 3 2 3 4 3 6 2 4 3
 0 0 5 2 5 3 4 5 3 0 6 3 0 3 2 5 2 4 6 5 2 3 4 2 2 2 3 2 4 0 5 3 6 2 4 2 6
 4 0 2 3 3 2 5 6 2 0 6 6 6 5 3 5 3 5 2 3 3 5 5 2 4 4 5 4 3 4 3 6 6 3 0 6 4
 3 6 6 3 3 3 3 0 0 3 4 3 